In [1]:
import os
import re
import sys
import math
import glob
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

try:
    import pydicom as dicom
except Exception as e:
    dicom = None
    print("WARNING: pydicom not available; image loading will be disabled.", repr(e))

try:
    import pylibjpeg  # optional
except Exception:
    pylibjpeg = None



device: cuda


In [2]:
IMAGES_DIR = "../input/rsna-2022-cervical-spine-fracture-detection/test_images"
TRAIN_IMAGES_PATH = "../input/rsna-2022-cervical-spine-fracture-detection/train_images"
TEST_IMAGES_PATH = "../input/rsna-2022-cervical-spine-fracture-detection/test_images"

TRAIN_CSV_PATH = "../input/rsna-2022-cervical-spine-fracture-detection/train.csv"
TEST_CSV_PATH = "../input/rsna-2022-cervical-spine-fracture-detection/test.csv"
SAMPLE_SUB_PATH = (
    "../input/rsna-2022-cervical-spine-fracture-detection/sample_submission.csv"
)

assert os.path.exists(TRAIN_CSV_PATH), f"Missing {TRAIN_CSV_PATH}"
assert os.path.exists(TEST_CSV_PATH), f"Missing {TEST_CSV_PATH}"
assert os.path.exists(SAMPLE_SUB_PATH), f"Missing {SAMPLE_SUB_PATH}"



In [3]:
segmentation_checkpoint = (
    "../input/effdet-models/axial_segmentation_effseg_132508-epoch-100.pth"
)
axial_det_checkpoint = (
    "../input/effdet-models/axial_detection_effdet_134352-epoch-52.pth"
)



In [4]:
df_test = pd.read_csv(TEST_CSV_PATH)
df_test.head()




,StudyInstanceUID,prediction_type,row_id
0,1.2.826.0.1.3680043.6200,patient_overall,1.2.826.0.1.3680043.6200_patient_overall
1,1.2.826.0.1.3680043.27262,patient_overall,1.2.826.0.1.3680043.27262_patient_overall
2,1.2.826.0.1.3680043.12351,patient_overall,1.2.826.0.1.3680043.12351_patient_overall
3,1.2.826.0.1.3680043.1363,patient_overall,1.2.826.0.1.3680043.1363_patient_overall
4,1.2.826.0.1.3680043.4859,patient_overall,1.2.826.0.1.3680043.4859_patient_overall


In [5]:
def rescale_img_to_hu(dcm_ds):
    return dcm_ds.pixel_array * float(getattr(dcm_ds, "RescaleSlope", 1.0)) + float(
        getattr(dcm_ds, "RescaleIntercept", 0.0)
    )


def normalize_hu(data):
    data = np.clip(data, a_min=-2242, a_max=2242) / 4484 + 0.5
    return data


def load_dicom(path):
    if dicom is None:
        raise RuntimeError("pydicom is not available in this environment.")
    ds = dicom.dcmread(path)
    img = rescale_img_to_hu(ds)
    pixel_spacing = float(ds.PixelSpacing[0]) if hasattr(ds, "PixelSpacing") else 1.0
    return img, pixel_spacing




In [6]:
class DcmDataSet(Dataset):
    def __init__(self, df, path, transforms=None):
        super().__init__()
        self.df = df
        self.path = path
        self.transforms = transforms
        self.len = len(self.df)

    def __getitem__(self, i):
        try:
            s = self.df.iloc[i]
            prev_s = self.df.iloc[i - 1] if i > 0 else s
            prev_s = s if prev_s.StudyInstanceUID != s.StudyInstanceUID else prev_s

            next_s = self.df.iloc[i + 1] if i < (self.len - 1) else s
            next_s = s if next_s.StudyInstanceUID != s.StudyInstanceUID else next_s

            rpath = os.path.join(
                self.path, s.StudyInstanceUID, f"{int(prev_s.Slice)}.dcm"
            )
            gpath = os.path.join(self.path, s.StudyInstanceUID, f"{int(s.Slice)}.dcm")
            bpath = os.path.join(
                self.path, s.StudyInstanceUID, f"{int(next_s.Slice)}.dcm"
            )

            g, pixel_spacing = load_dicom(gpath)
            r, _ = load_dicom(rpath)
            b, _ = load_dicom(bpath)

            img = np.stack((r, g, b))
            img = normalize_hu(img)
            if self.transforms is not None:
                img = self.transforms(img)
        except Exception:
            return None, None

        return img, pixel_spacing

    def __len__(self):
        return self.len


class DataTransform(nn.Module):
    def __init__(self, image_size=512):
        super().__init__()
        self.image_size = image_size

    def forward(self, x):
        x = torch.as_tensor(x, dtype=torch.float32)
        return x




In [7]:
def safe_collate(batch):
    batch = [b for b in batch if b[0] is not None]
    if len(batch) == 0:
        return torch.empty((0, 3, 512, 512)), torch.empty((0,))
    xs, ps = zip(*batch)
    return torch.stack(xs, 0), torch.as_tensor(ps, dtype=torch.float32)


dl = None  # not used in baseline




In [8]:
def get_axial_segmentation_model(checkpoint):
    raise ModuleNotFoundError(
        "efficientunet/weights not available in this environment; baseline uses priors only."
    )


seg_model = None




In [9]:
def get_axial_detection_model(checkpoint, image_size=512):
    raise ModuleNotFoundError(
        "effdet/weights not available in this environment; baseline uses priors only."
    )


det_model = None




In [10]:
def get_axial_boundary_from_segmentation(
    seg, pixel_spacing, throw=100, tol=0.2, max_mm=100
):
    image_size = seg.shape[0]
    min_size = min(image_size, max_mm / pixel_spacing)

    rows, columns = seg.nonzero(as_tuple=True)
    rows, _ = torch.sort(rows)
    columns, _ = torch.sort(columns)

    throw = min(len(rows) // 2, throw)

    if len(rows) == 0:
        return torch.tensor([0, 0, image_size, image_size], device=seg.device)

    xmin, xmax = columns[throw], columns[-throw]
    ymin, ymax = rows[throw], rows[-throw]

    w = (xmax - xmin) * (1 + tol)
    h = (ymax - ymin) * (1 + tol)
    new_size = max(w, h, torch.as_tensor(min_size, device=seg.device))
    new_size = min(image_size, float(new_size))

    xcenter, ycenter = (xmax + xmin) / 2, (ymax + ymin) / 2

    xmin = torch.minimum(
        torch.tensor(image_size - new_size, device=seg.device), xcenter - new_size / 2
    ).clamp(min=0)
    ymin = torch.minimum(
        torch.tensor(image_size - new_size, device=seg.device), ycenter - new_size / 2
    ).clamp(min=0)

    return torch.stack([xmin, ymin, xmin + new_size, ymin + new_size])




In [11]:
def predict_seg(x, model, img_size=256):
    raise RuntimeError("Segmentation model not available in this baseline.")




In [12]:
def get_axial_boundary(segs, pixel_spacings, seg_img_size=256):
    raise RuntimeError("Segmentation model not available in this baseline.")




In [13]:
def predict_det(x, model):
    raise RuntimeError("Detection model not available in this baseline.")


def crop_resize_images(imgs_tensor, boundary_list, img_size=512):
    raise RuntimeError("Detection model not available in this baseline.")


def get_original_bbox(bbox, boundary):
    raise RuntimeError("Detection model not available in this baseline.")




In [14]:
def get_bbox_class(seg, bbox):
    raise RuntimeError("Detection model not available in this baseline.")




In [15]:
def get_bbox_class_list(seg_list, seg_bboxes):
    raise RuntimeError("Detection model not available in this baseline.")




In [16]:
def get_class_score(scores, class_list, eps=1e-2):
    raise RuntimeError("Detection model not available in this baseline.")


def check_detection_result(det_result, img_size=512.0, threshold=0.2):
    raise RuntimeError("Detection model not available in this baseline.")




In [17]:
def cal_loss(prob, label):
    pos_weight = np.array([14, 2, 2, 2, 2, 2, 2, 2])
    neg_weight = np.array([7, 1, 1, 1, 1, 1, 1, 1])
    prob = np.clip(prob, 1e-6, 1 - 1e-6)
    score = pos_weight * label * np.log(prob) + neg_weight * (1 - label) * np.log(
        1 - prob
    )
    weight_total = pos_weight * label + neg_weight * (1 - label)
    return -score.sum(axis=1) / weight_total.sum(axis=1)




In [18]:
def _weighted_logloss_constant(
    y: np.ndarray, p: float, w_pos: float, w_neg: float
) -> float:
    """
    Mean weighted binary log loss for a single label with constant probability p, matching
    the competition-style per-row weighting (here implemented as w_pos for positive rows and
    w_neg for negative rows for that label).
    """
    p = float(np.clip(p, 1e-6, 1 - 1e-6))
    y = y.astype(np.float64)
    loss = -(y * np.log(p) + (1.0 - y) * np.log(1.0 - p))
    weights = y * float(w_pos) + (1.0 - y) * float(w_neg)
    return float(np.sum(weights * loss) / np.sum(weights))


def _choose_constant_p_weighted_closed_form(
    y: np.ndarray, w_pos: float, w_neg: float, a0: float = 0.5
) -> float:
    """
    Score-moving change:
    - For a constant predictor under weighted log loss, the optimum is:
        p* = (w_pos * n_pos) / (w_pos*n_pos + w_neg*n_neg)
      We add a tiny symmetric pseudo-count (a0, a0) for stability on small N.
    This directly targets the evaluation metric while keeping the same "priors-only constant" core logic.
    """
    y = y.astype(np.int64)
    n_pos = float(y.sum())
    n = float(len(y))
    n_neg = n - n_pos

    num = w_pos * n_pos + w_pos * a0
    den = (w_pos * n_pos + w_neg * n_neg) + (w_pos + w_neg) * a0
    p = num / max(den, 1e-12)
    return float(np.clip(p, 1e-4, 1 - 1e-4))


def predict_priors(train_csv_path: str):
    """
    Priors-only baseline (core logic preserved): output constant probabilities per label.

    Score-moving change:
    - Choose each label's constant by minimizing the competition-weighted log loss for that label.
      This is a minimal calibration improvement and should reduce the leaderboard loss.
    - Keep patient_overall as its own constant (still), because it is heavily weighted.
    """
    df_train = pd.read_csv(train_csv_path)
    cols = ["patient_overall"] + [f"C{i}" for i in range(1, 8)]

    # Competition weights (from RSNA 2022): patient_overall has higher weight.
    # Implemented here as per-row weights (pos/neg) used in many reference implementations.
    w_pos = {
        "patient_overall": 14.0,
        "C1": 2.0,
        "C2": 2.0,
        "C3": 2.0,
        "C4": 2.0,
        "C5": 2.0,
        "C6": 2.0,
        "C7": 2.0,
    }
    w_neg = {
        "patient_overall": 7.0,
        "C1": 1.0,
        "C2": 1.0,
        "C3": 1.0,
        "C4": 1.0,
        "C5": 1.0,
        "C6": 1.0,
        "C7": 1.0,
    }

    pri = {}
    for c in cols:
        y = df_train[c].values
        pri[c] = _choose_constant_p_weighted_closed_form(
            y, w_pos=w_pos[c], w_neg=w_neg[c], a0=0.5
        )

        # Keep a tiny safety check (no behavior change intended) to ensure the closed-form isn't worse on train.
        # If numeric issues ever arise, fall back to the better of (prevalence, closed-form) under weighted loss.
        p_prev = float(np.clip((y.mean()), 1e-4, 1 - 1e-4))
        l_cf = _weighted_logloss_constant(y, pri[c], w_pos[c], w_neg[c])
        l_prev = _weighted_logloss_constant(y, p_prev, w_pos[c], w_neg[c])
        if not np.isfinite(l_cf) or (l_prev + 1e-12) < l_cf:
            pri[c] = p_prev

    return pri


priors = predict_priors(TRAIN_CSV_PATH)
priors



{'patient_overall': 0.6352941176470588,
 'C1': 0.18344519015659955,
 'C2': 0.2537634408602151,
 'C3': 0.08941176470588236,
 'C4': 0.10722610722610723,
 'C5': 0.19955654101995565,
 'C6': 0.2905263157894737,
 'C7': 0.2832980972515856}

In [19]:
study_ids = df_test["StudyInstanceUID"].unique()
df_patient_pred = pd.DataFrame({"StudyInstanceUID": study_ids})
for k in ["patient_overall"] + [f"C{i}" for i in range(1, 8)]:
    df_patient_pred[k] = float(priors[k])

df_patient_pred = df_patient_pred.set_index("StudyInstanceUID")
df_patient_pred.head()



,patient_overall,C1,C2,C3,C4,C5,C6,C7
StudyInstanceUID,,,,,,,,
1.2.826.0.1.3680043.6200,0.635294,0.183445,0.253763,0.089412,0.107226,0.199557,0.290526,0.283298
1.2.826.0.1.3680043.27262,0.635294,0.183445,0.253763,0.089412,0.107226,0.199557,0.290526,0.283298
1.2.826.0.1.3680043.12351,0.635294,0.183445,0.253763,0.089412,0.107226,0.199557,0.290526,0.283298
1.2.826.0.1.3680043.1363,0.635294,0.183445,0.253763,0.089412,0.107226,0.199557,0.290526,0.283298
1.2.826.0.1.3680043.4859,0.635294,0.183445,0.253763,0.089412,0.107226,0.199557,0.290526,0.283298


In [20]:
pass



In [21]:
if df_test.iloc[0].row_id == "1.2.826.0.1.3680043.10197_C1":
    df_test = pd.DataFrame(
        {
            "row_id": [
                "1.2.826.0.1.3680043.22327_C1",
                "1.2.826.0.1.3680043.25399_C1",
                "1.2.826.0.1.3680043.5876_patient_overall",
            ],
            "StudyInstanceUID": [
                "1.2.826.0.1.3680043.22327",
                "1.2.826.0.1.3680043.25399",
                "1.2.826.0.1.3680043.5876",
            ],
            "prediction_type": ["C1", "C1", "patient_overall"],
        }
    )

df_test.head()



,StudyInstanceUID,prediction_type,row_id
0,1.2.826.0.1.3680043.6200,patient_overall,1.2.826.0.1.3680043.6200_patient_overall
1,1.2.826.0.1.3680043.27262,patient_overall,1.2.826.0.1.3680043.27262_patient_overall
2,1.2.826.0.1.3680043.12351,patient_overall,1.2.826.0.1.3680043.12351_patient_overall
3,1.2.826.0.1.3680043.1363,patient_overall,1.2.826.0.1.3680043.1363_patient_overall
4,1.2.826.0.1.3680043.4859,patient_overall,1.2.826.0.1.3680043.4859_patient_overall


In [22]:
df_sub = df_test.copy()
df_sub = (
    df_sub.set_index("StudyInstanceUID").join(df_patient_pred, how="left").reset_index()
)

for k in ["patient_overall"] + [f"C{i}" for i in range(1, 8)]:
    df_sub[k] = df_sub[k].fillna(float(priors[k]))

# Keep semantics identical: per-row probability is the constant for that prediction_type.
df_sub["fractured"] = df_sub.apply(lambda r: float(r[r["prediction_type"]]), axis=1)
df_sub[["row_id", "fractured"]].head()



,row_id,fractured
0,1.2.826.0.1.3680043.6200_patient_overall,0.635294
1,1.2.826.0.1.3680043.27262_patient_overall,0.635294
2,1.2.826.0.1.3680043.12351_patient_overall,0.635294
3,1.2.826.0.1.3680043.1363_patient_overall,0.635294
4,1.2.826.0.1.3680043.4859_patient_overall,0.635294


In [23]:
sub_path = "submission.csv"
df_sub[["row_id", "fractured"]].to_csv(sub_path, index=False)

df_sample = pd.read_csv(SAMPLE_SUB_PATH)
assert list(df_sample.columns) == ["row_id", "fractured"]
assert df_sub.shape[0] == df_sample.shape[0], (df_sub.shape, df_sample.shape)
print("Wrote:", sub_path, "rows:", df_sub.shape[0])
print(df_sub.head())

Wrote: submission.csv rows: 14536
            StudyInstanceUID  prediction_type  \
0   1.2.826.0.1.3680043.6200  patient_overall   
1  1.2.826.0.1.3680043.27262  patient_overall   
2  1.2.826.0.1.3680043.12351  patient_overall   
3   1.2.826.0.1.3680043.1363  patient_overall   
4   1.2.826.0.1.3680043.4859  patient_overall   

                                      row_id  patient_overall        C1  \
0   1.2.826.0.1.3680043.6200_patient_overall         0.635294  0.183445   
1  1.2.826.0.1.3680043.27262_patient_overall         0.635294  0.183445   
2  1.2.826.0.1.3680043.12351_patient_overall         0.635294  0.183445   
3   1.2.826.0.1.3680043.1363_patient_overall         0.635294  0.183445   
4   1.2.826.0.1.3680043.4859_patient_overall         0.635294  0.183445   

         C2        C3        C4        C5        C6        C7  fractured  
0  0.253763  0.089412  0.107226  0.199557  0.290526  0.283298   0.635294  
1  0.253763  0.089412  0.107226  0.199557  0.290526  0.283298   0.6352